## 1. Import & trỏ `sys.path` tới module project

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="albumentations")
warnings.filterwarnings("ignore", category=UserWarning, module="rtdl_num_embeddings")

import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd

# --- Tìm thư mục project (chứa runner.py / config.py) ---
# Trên Kaggle: upload thư mục 'mid_fusion_lite' thành Dataset rồi thêm đường dẫn vào đây.
PROJECT_DIR_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/kaggle/input/mid-fusion"),
]
PROJECT_DIR = next((c for c in PROJECT_DIR_CANDIDATES if (c / "runner.py").exists()), None)
assert PROJECT_DIR is not None, (
    "Khong tim thay module project (runner.py). Them duong dan Kaggle Dataset chua code "
    "vao PROJECT_DIR_CANDIDATES."
)
sys.path.insert(0, str(PROJECT_DIR))
print("Project dir:", PROJECT_DIR)

from config import (
    MidFusionLiteConfig, resolve_data_paths,
    BASE_NUMERIC_COLS, ENGINEERED_NUMERIC_COLS,
    UGLY_DUCKLING_NUMERIC_COLS, UGLY_DUCKLING_SUMMARY_COLS,
    CATEGORICAL_COUNT_COLS, NUM_CAT_INTERACTION_COLS, DEFAULT_CATEGORICAL_COLS,
)
from runner import run_mid_fusion_cv

IS_KAGGLE = Path("/kaggle/input").exists()
print("Moi truong:", "KAGGLE" if IS_KAGGLE else "LOCAL")

Project dir: /kaggle/input/datasets/trnhintmh03/isic-04-07
Moi truong: KAGGLE


## 2. Dò dữ liệu ISIC 2024 (Kaggle / local)

In [2]:
def resolve_notebook_data_paths(config):
    """Kaggle: quét /kaggle/input tìm train-metadata.csv. Local: dùng resolve_data_paths."""
    if IS_KAGGLE:
        for root, _dirs, files in os.walk("/kaggle/input"):
            if "train-metadata.csv" in files:
                r = Path(root)
                print("Kaggle dataset root:", r)
                return config.with_updates(
                    train_metadata_path=str(r / "train-metadata.csv"),
                    test_metadata_path=str(r / "test-metadata.csv"),
                    sample_submission_path=str(r / "sample_submission.csv"),
                    train_hdf5_path=str(r / "train-image.hdf5"),
                    test_hdf5_path=str(r / "test-image.hdf5"),
                )
        raise FileNotFoundError("Khong tim thay train-metadata.csv trong /kaggle/input.")
    return resolve_data_paths(config)

## 3. Công tắc ablation + định nghĩa nhóm feature

`BASE_NUMERIC_COLS` (TBP thô) **luôn được giữ** làm baseline. 4 nhóm bật/tắt độc lập.
Ablation bằng **chọn cột**: ta build danh sách cột ứng viên theo cờ, rồi truyền vào
`config.numeric_cols` / `config.categorical_cols`. Runner tự loại cột không tồn tại.

In [3]:
# === CONG TAC NHOM FEATURE DOMAIN Y TE (bat/tat de ablation) ===
FEATURE_FLAGS = {
    "use_engineered_abcd":     True,   # ABCD engineered (hinh thai/mau sac)
    "use_patient_norm":        True,   # Ugly-Duckling loi: *_patient_norm + *_patient_variation
    "use_ugly_duckling_adv":   True,   # UD nang cao: ud_*, LOF, *_anatomical_zscore, area-agg
    "use_categorical_context": True,   # categorical embeds + counts + site-norm + num x cat
}

# --- Dan xuat cac nhom cot tu hang so config (khong can chay FE truoc) ---
_SRC = list(BASE_NUMERIC_COLS) + list(ENGINEERED_NUMERIC_COLS)

_anatomical_zscore = [c for c in ENGINEERED_NUMERIC_COLS if c.endswith("_anatomical_zscore")]
_patient_variation = [c for c in ENGINEERED_NUMERIC_COLS if c.endswith("_patient_variation")]
_area_agg = [c for c in ["tbp_lv_areaMM2_patient", "tbp_lv_areaMM2_bp"] if c in ENGINEERED_NUMERIC_COLS]
_lof = [c for c in ["patient_lof_score"] if c in ENGINEERED_NUMERIC_COLS]
_non_abcd = set(_anatomical_zscore) | set(_patient_variation) | set(_area_agg) | set(_lof)

GROUP_BASE = list(BASE_NUMERIC_COLS)                                             # luon giu
GROUP_ABCD = [c for c in ENGINEERED_NUMERIC_COLS if c not in _non_abcd]
GROUP_PATIENT_NORM = [f"{c}_patient_norm" for c in _SRC] + _patient_variation
GROUP_UD_ADV = list(UGLY_DUCKLING_NUMERIC_COLS) + list(UGLY_DUCKLING_SUMMARY_COLS) \
             + _lof + _anatomical_zscore + _area_agg
GROUP_CAT_NUMERIC = list(CATEGORICAL_COUNT_COLS) \
                  + ["count_per_patient", "is_single_lesion_patient"] \
                  + [f"{c}_patient_site_general_norm" for c in _SRC] \
                  + list(NUM_CAT_INTERACTION_COLS)
GROUP_CAT_EMBED = list(DEFAULT_CATEGORICAL_COLS)   # sex, anatom_site_general, tbp_tile_type, ...


def build_column_candidates(flags):
    """Tra ve (numeric_candidates, categorical_candidates) theo cac co feature.
    Day la SUPERSET ung vien; runner.select_available_columns se loai cot vang mat."""
    num = list(GROUP_BASE)
    if flags.get("use_engineered_abcd", True):
        num += GROUP_ABCD
    if flags.get("use_patient_norm", True):
        num += GROUP_PATIENT_NORM
    if flags.get("use_ugly_duckling_adv", True):
        num += GROUP_UD_ADV
    if flags.get("use_categorical_context", True):
        num += GROUP_CAT_NUMERIC
        cat = list(GROUP_CAT_EMBED)
    else:
        cat = []
    # de-dup giu thu tu
    num = list(dict.fromkeys(num))
    return num, cat


for _name, _g in [("BASE", GROUP_BASE), ("ABCD", GROUP_ABCD), ("PATIENT_NORM", GROUP_PATIENT_NORM),
                  ("UD_ADV", GROUP_UD_ADV), ("CAT_NUMERIC", GROUP_CAT_NUMERIC), ("CAT_EMBED", GROUP_CAT_EMBED)]:
    print(f"{_name:14s}: {len(_g):4d} cot (ung vien)")
_n, _c = build_column_candidates(FEATURE_FLAGS)
print(f"\n=> Cau hinh hien tai: {len(_n)} numeric + {len(_c)} categorical (truoc khi runner loc)")

BASE          :   34 cot (ung vien)
ABCD          :   69 cot (ung vien)
PATIENT_NORM  :  115 cot (ung vien)
UD_ADV        :  233 cot (ung vien)
CAT_NUMERIC   :  289 cot (ung vien)
CAT_EMBED     :    5 cot (ung vien)

=> Cau hinh hien tai: 740 numeric + 5 categorical (truoc khi runner loc)


## 4. Siêu tham số TabM + builder cấu hình

Đưa toàn bộ hyper-param TabM (Trial 53) vào `MidFusionLiteConfig`. Với `negative_ratio=100`
và `pos_weight=2.648`, runner tự đặt `init_bias = -log(100)` cho `tabular_only_head`.

In [4]:
SMOKE = False   # True = chay nhanh kiem thu (2 fold, 4 epoch, 1 fold_index)

OUTPUT_DIR = "/kaggle/working/tabm_output" if IS_KAGGLE else "./mid_fusion_lite_outputs/tabm_only"

def build_config(flags):
    num_cols, cat_cols = build_column_candidates(flags)
    cfg = MidFusionLiteConfig(
        preset_name="tabm_only_ablation",
        seed=42,
        # --- Modality: chi nhanh bang, train end-to-end (khong staged) ---
        modality_mode="tabular_only",
        use_staged_pretraining=False,
        use_tabm_tabular=True,
        fusion_mode="concat",           # khong dung trong tabular_only, dat cho nhat quan
        # --- Cross-validation (giong notebook cu: StratifiedGroupKFold) ---
        n_splits=5,
        fold_col=None,                  # ep dung StratifiedGroupKFold theo patient_id
        fold_indices=None,              # None = chay het 5 fold
        # --- Feature engineering: SINH DAY DU, ablation lam qua numeric_cols/categorical_cols ---
        use_feature_engineering=True,
        use_contextual_group_norm_features=True,
        use_ugly_duckling_features=True,
        use_anatomical_zscore_features=True,
        use_area_aggregation_features=True,
        numeric_cols=num_cols,
        categorical_cols=cat_cols,
        # --- TabM core (Trial 53) ---
        tabm_arch_type="tabm",
        tabm_k=32,
        tabm_n_blocks=2,
        tabm_d_block=256,
        tab_hidden_dim=256,
        n_bins=48,
        d_embedding=32,
        tab_dropout=0.3,
        # --- Optim / schedule ---
        lr=1e-3,
        weight_decay=1.32e-4,
        warmup_epochs=5,
        min_lr=1e-6,
        gradient_clipping_norm=1.0,
        label_smoothing=0.01,
        epochs=100,
        patience=15,
        # --- Batch & imbalance ---
        train_batch_size=512,
        valid_batch_size=512,
        negative_ratio=100,             # sampler 1:100 -> init_bias = -log(100)
        pos_weight=10,               # co dinh (khong None) -> dung truc tiep
        # --- He thong ---
        num_workers=4,                  # Kaggle-safe
        use_amp=True,
        save_checkpoints=True,
        output_dir=OUTPUT_DIR,
    )
    if SMOKE:
        cfg = cfg.with_updates(epochs=4, patience=3, warmup_epochs=1, n_splits=2, fold_indices=[0])
    return resolve_notebook_data_paths(cfg)

print("SMOKE =", SMOKE, "| output_dir =", OUTPUT_DIR)

SMOKE = False | output_dir = /kaggle/working/tabm_output


## 5. Wrapper chạy 1 cấu hình + ghi log CSV

In [5]:
RESULTS_CSV = "ablation_results.csv"

def run_one(flags, tag="manual", log_path=RESULTS_CSV):
    cfg = build_config(flags)
    print("=" * 70)
    print(f"  RUN [{tag}] flags = {flags}")
    print("=" * 70)
    results = run_mid_fusion_cv(cfg)

    cv_pauc = results.get("cv_pauc")
    fold_paucs = [f["fold_valid_pauc"] for f in results.get("fold_metrics", [])]
    row = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "tag": tag,
        **{k: int(v) for k, v in flags.items()},
        "cv_pauc": round(float(cv_pauc), 6) if cv_pauc is not None else None,
        "mean_fold_pauc": round(float(np.mean(fold_paucs)), 6) if fold_paucs else None,
        "n_numeric": len(cfg.numeric_cols),
        "n_categorical": len(cfg.categorical_cols),
        "fold_paucs": json.dumps([round(float(s), 5) for s in fold_paucs]),
    }
    pd.DataFrame([row]).to_csv(log_path, mode="a", header=not Path(log_path).exists(), index=False)
    print(f"\n[{tag}] CV pAUC = {row['cv_pauc']} | mean fold pAUC = {row['mean_fold_pauc']}")
    return results, row

## 6. Chạy cấu hình hiện tại (tất cả feature bật)

In [6]:
results_full, row_full = run_one(FEATURE_FLAGS, tag="full")

# Xuat submission cho Kaggle (bat buoc ten submission.csv o /kaggle/working)
import shutil
sub_src = results_full.get("submission_path")
if sub_src and os.path.exists(sub_src):
    print("Submission goc:", sub_src)
    if IS_KAGGLE:
        shutil.copy(sub_src, "/kaggle/working/submission.csv")
        print("Da copy -> /kaggle/working/submission.csv")

# Liet ke checkpoint da luu (mid_fusion_lite_fold_{i}.pt -> nap duoc o Stage-3 fusion)
ckpts = sorted(Path(OUTPUT_DIR).rglob("*.pt"))
print(f"\n{len(ckpts)} checkpoint:")
for p in ckpts:
    print("  -", p)

Kaggle dataset root: /kaggle/input/competitions/isic-2024-challenge
  RUN [full] flags = {'use_engineered_abcd': True, 'use_patient_norm': True, 'use_ugly_duckling_adv': True, 'use_categorical_context': True}
Running preset: tabm_only_ablation
Device: CUDA
Output directory: /kaggle/working/tabm_output
Fold column 'None' not found. Generating fold splits dynamically using StratifiedGroupKFold.
Selected folds: [0, 1, 2, 3, 4]
Fold 1/5
Numeric features: 610 | Categorical features: 5
  [end-to-end (both)] sampler 1:100 -> pos_weight_sampled=10.000
Epoch 01 | train_loss=0.49447 | valid_loss=0.15857 | valid_auroc=0.77641 | valid_pauc=0.07160
Epoch 02 | train_loss=0.37221 | valid_loss=0.09396 | valid_auroc=0.93798 | valid_pauc=0.15529
Epoch 03 | train_loss=0.33309 | valid_loss=0.07420 | valid_auroc=0.94821 | valid_pauc=0.15959
Epoch 04 | train_loss=0.31943 | valid_loss=0.09575 | valid_auroc=0.95500 | valid_pauc=0.16370
Epoch 05 | train_loss=0.30881 | valid_loss=0.09550 | valid_auroc=0.95053 |